In [1]:
import numpy as np
import cupy as cp
import scipy
from typing import Callable, Any
import quadpy
import time
import math

num_points = 10**5
points = np.random.randn(num_points, 3)
magnitudes = np.linalg.norm(points, axis=1, keepdims=True)
normalized_points = points / magnitudes
hull = scipy.spatial.ConvexHull(normalized_points)


# cube = np.array(
#     [
#         [0, 0, 0],
#         [1, 0, 0],
#         [1, 1, 0],
#         [0, 1, 0],
#         [0, 0, 1],
#         [1, 0, 1],
#         [1, 1, 1],
#         [0, 1, 1],
#     ]
# )
# hull = scipy.spatial.ConvexHull(cube)
print(f"Hull has {len(hull.simplices)} simplices.")

Hull has 199996 simplices.


In [2]:
def hull_surface_integral(
    function: Callable,
    hull: scipy.spatial.ConvexHull,
    scheme: Any | None = None,
    use_gpu: bool = False,
) -> np.ndarray | cp.ndarray:
    """Compute the integral of a function over the surface of a convex hull. It
    is assumed that the function returns a scalar output, i.e. maps R^n -> R.

    function: The function to be integrated. This function must be vectorized
    to accept arguments of shape

    N x n

    where n-1 is the number of dimensions of the simplical facets of the
    surface (alternatively, n is the number of dimensions of the ambient space
    in which the surface lies. For example, for the surface of a sphere, n=3)

    hull: The convex hull object that defines the surface.
    scheme: Integration scheme to be used.
    use_gpu: If true, use cupy instead of numpy
    """
    # Pick appropriate array module
    xp = cp if use_gpu else np

    # Pick integration scheme
    # For future developers: one can use Cayley-Menger determinants for d > 2
    num_dimensions = hull.points.shape[1] - 1
    if num_dimensions > 2:
        raise NotImplementedError(
            "Integration of surfaces in d > 2 dimensions"
            "not currently supported."
        )

    if scheme is None:
        # The second parameter to quadpy's scheme method here is about
        # accuracy of the scheme, not the number of spatial dimensions.
        scheme = quadpy.tn.grundmann_moeller(num_dimensions, 3)

    barycentric_weights = xp.asarray(scheme.points)
    weights = xp.asarray(scheme.weights)

    # Generate integration points
    num_simplices = len(hull.simplices)
    num_weights = len(weights)
    simplical_points = xp.asarray(hull.points)[
        xp.asarray(hull.simplices)
    ].transpose(0, 2, 1)
    integration_points = simplical_points @ barycentric_weights
    reshaped_points = integration_points.transpose(0, 2, 1).reshape(
        -1, num_dimensions + 1
    )

    # Find simplex volumes using cross product
    v1s = simplical_points[:, :, 1] - simplical_points[:, :, 0]
    v2s = simplical_points[:, :, 2] - simplical_points[:, :, 0]
    cross_products = xp.cross(v1s, v2s)
    areas = 0.5 * xp.sqrt(xp.sum(cross_products**2, axis=1))

    # Output
    function_output = function(reshaped_points)
    weighted_output = (
        function_output
        * xp.tile(weights, num_simplices)
        * xp.repeat(areas, num_weights)
    )
    integral = xp.sum(weighted_output)
    return integral

In [3]:
def hull_surface_integral_vector(
    function: Callable,
    hull: scipy.spatial.ConvexHull,
    scheme: Any | None = None,
    use_gpu: bool = False,
) -> np.ndarray | cp.ndarray:
    """Compute the integral of a function over the surface of a convex hull. It
    is assumed that the function returns a vector output, i.e. maps R^3 -> R^3.

    function: The function to be integrated. This function must be vectorized
    to accept arguments of shape

    N x n

    where n-1 is the number of dimensions of the simplical facets of the
    surface (alternatively, n is the number of dimensions of the ambient space
    in which the surface lies. For example, for the surface of a sphere, n=3)

    hull: The convex hull object that defines the surface.
    scheme: Integration scheme to be used.
    use_gpu: If true, use cupy instead of numpy
    """
    # Pick appropriate array module
    xp = cp if use_gpu else np

    # Pick integration scheme
    # For future developers: one can use Cayley-Menger determinants for d > 2
    num_dimensions = hull.points.shape[1] - 1
    if num_dimensions > 2:
        raise NotImplementedError(
            "Integration of surfaces in d > 2 dimensions"
            "not currently supported."
        )

    if scheme is None:
        # The second parameter to quadpy's scheme method here is about
        # accuracy of the scheme, not the number of spatial dimensions.
        scheme = quadpy.tn.grundmann_moeller(num_dimensions, 3)

    barycentric_weights = xp.asarray(scheme.points)
    weights = xp.asarray(scheme.weights)
    points = xp.asarray(hull.points)
    vertices = xp.asarray(hull.vertices)
    centroid = xp.mean(points[vertices], axis=0)
    simplices = xp.asarray(hull.simplices)

    # Generate integration points
    num_simplices = len(hull.simplices)
    num_weights = len(weights)
    simplical_points = points[simplices].transpose(0, 2, 1)
    integration_points = simplical_points @ barycentric_weights
    reshaped_points = integration_points.transpose(0, 2, 1).reshape(
        -1, num_dimensions + 1
    )

    # Find simplex volumes using cross product
    v1s = simplical_points[:, :, 1] - simplical_points[:, :, 0]
    v2s = simplical_points[:, :, 2] - simplical_points[:, :, 0]
    cross_products = xp.cross(v1s, v2s)
    unit_normals = cross_products / xp.linalg.norm(
        cross_products, axis=1, keepdims=True
    )
    radial_points = simplical_points[:, :, 0] - centroid
    dot_products = xp.sum(radial_points * unit_normals, axis=1)
    orientation_signs = xp.sign(dot_products)
    areas = 0.5 * xp.sqrt(xp.sum(cross_products**2, axis=1))

    # Output
    function_output = function(reshaped_points)
    integrand = xp.sum(
        function_output.reshape(num_simplices, num_weights, 3)
        * unit_normals[:, None, :],
        axis=2,
    ).reshape(num_simplices * num_weights)

    weighted_output = (
        integrand
        * xp.tile(weights, num_simplices)
        * xp.repeat(areas, num_weights)
        * xp.repeat(orientation_signs, num_weights)
    )
    integral = xp.sum(weighted_output)
    return integral

In [4]:
def simplex_integral(
    function: Callable,
    simplices: np.ndarray,
    scheme: Any | None = None,
    use_gpu: bool = False,
) -> np.ndarray | cp.ndarray:
    """Compute the integral of a function over a (collection of) simplex/ices.

    function: The function to be integrated. This function must be vectorized
    to accept arguments of shape

    N x (n+1) x n

    where n is the number of dimensions of the simplices

    simplices: The simplices with shape as described above.
    scheme: Integration scheme to be used.
    use_gpu: If true, use cupy instead of numpy
    """
    # Pick appropriate array module
    xp = cp.get_array_module(simplices)

    # Pick integration scheme
    num_dimensions = simplices.shape[2]
    if scheme is None:
        # The second parameter to quadpy's scheme method here is about
        # accuracy of the scheme, not the number of spatial dimensions.
        scheme = quadpy.tn.grundmann_moeller(num_dimensions, 3)

    barycentric_weights = xp.asarray(scheme.points)
    weights = xp.asarray(scheme.weights)

    # Find simplex volumes using Cayley-Menger determinants
    contents = xp.abs(
        xp.linalg.det(simplices[:, 1:, :] - simplices[:, 0, None, :])
        / math.factorial(num_dimensions)
    )

    # Generate integration points
    integration_points = simplices @ barycentric_weights
    reshaped_points = integration_points.transpose(0, 2, 1).reshape(
        -1, num_dimensions + 1
    )

    # Output
    function_output = function(reshaped_points)
    weighted_output = (
        function_output
        * xp.tile(weights, num_simplices)
        * xp.repeat(areas, num_weights)
    )
    integral = xp.sum(weighted_output)
    return integral

In [5]:
def identity_vector(x):
    return (
        np.repeat(x[:, 0] ** 2, 3).reshape(len(x), 3)
        * x
        / np.linalg.norm(x, axis=1, keepdims=True)
    )


def identity_vector_cp(x):
    return (
        cp.repeat(x[:, 0] ** 2, 3).reshape(len(x), 3)
        * x
        / cp.linalg.norm(x, axis=1, keepdims=True)
    )


s = time.perf_counter()
integral = hull_surface_integral_vector(identity_vector, hull, use_gpu=False)
e = time.perf_counter()
print(f"Time elapsed CPU: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area/3}")
print(f"Error one: {hull.area/3 - integral}")
print("---")
s = time.perf_counter()
integral = hull_surface_integral_vector(identity_vector_cp, hull, use_gpu=True)
e = time.perf_counter()
print(f"Time elapsed GPU: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area/3}")
print(f"Error one: {hull.area/3 - integral}")
print("---")
s = time.perf_counter()
integral = hull_surface_integral_vector(identity_vector_cp, hull, use_gpu=True)
e = time.perf_counter()
print(f"Time elapsed: GPU second: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area/3}")
print(f"Error one: {hull.area/3 - integral}")
print("---")

Time elapsed CPU: 0.3555064257234335
Calculated area: 4.188112459642662
Actual area: 4.188538301064719
Error one: 0.0004258414220572604
---
Time elapsed GPU: 0.44419262930750847
Calculated area: 4.1881124596426655
Actual area: 4.188538301064719
Error one: 0.0004258414220537077
---
Time elapsed: GPU second: 0.0018991455435752869
Calculated area: 4.1881124596426655
Actual area: 4.188538301064719
Error one: 0.0004258414220537077
---


In [6]:
# Irrotational field
def irrotational(input):
    x, y, _ = input[:, 0], input[:, 1], input[:, 2]
    z = np.zeros(np.shape(x))
    output = np.column_stack((-y, x, z))
    return output


def radial_vector(input):
    return input


s = time.perf_counter()
integral = hull_surface_integral_vector(radial_vector, hull, use_gpu=False)
e = time.perf_counter()
print(f"Time elapsed CPU: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area}")
print(f"Error one: {np.abs(hull.area - integral)}")
print("---")

Time elapsed CPU: 0.21874921768903732
Calculated area: 12.564857620723147
Actual area: 12.565614903194158
Error one: 0.0007572824710102566
---


In [7]:
cube = np.array(
    [
        [0, 0, 0],
        [1, 0, 0],
        [1, 1, 0],
        [0, 1, 0],
        [0, 0, 1],
        [1, 0, 1],
        [1, 1, 1],
        [0, 1, 1],
    ]
)
hull = scipy.spatial.ConvexHull(cube)


def outward_normal(input):
    pos_x = np.where(np.isclose(input[:, 0], 1.0))
    neg_x = np.where(np.isclose(input[:, 0], 0.0))
    pos_y = np.where(np.isclose(input[:, 1], 1.0))
    neg_y = np.where(np.isclose(input[:, 1], 0.0))
    pos_z = np.where(np.isclose(input[:, 2], 1.0))
    neg_z = np.where(np.isclose(input[:, 2], 0.0))
    input[pos_x] = np.tile(np.array([1.0, 0.0, 0.0]), len(pos_x[0])).reshape(
        len(pos_x[0]), 3
    )
    input[neg_x] = np.tile(np.array([-1.0, 0.0, 0.0]), len(neg_x[0])).reshape(
        len(neg_x[0]), 3
    )
    input[pos_y] = np.tile(np.array([0.0, 1.0, 0.0]), len(pos_y[0])).reshape(
        len(pos_y[0]), 3
    )
    input[neg_y] = np.tile(np.array([0.0, -1.0, 0.0]), len(neg_y[0])).reshape(
        len(neg_y[0]), 3
    )
    input[pos_z] = np.tile(np.array([0.0, 0.0, 1.0]), len(pos_z[0])).reshape(
        len(pos_z[0]), 3
    )
    input[neg_z] = np.tile(np.array([0.0, 0.0, -1.0]), len(neg_z[0])).reshape(
        len(neg_z[0]), 3
    )
    return input


s = time.perf_counter()
integral = hull_surface_integral_vector(outward_normal, hull, use_gpu=False)
e = time.perf_counter()
print(f"Time elapsed CPU: {e-s}")
print(f"Calculated area: {integral}")
print(f"Actual area: {hull.area}")
print(f"Error one: {np.abs(hull.area - integral)}")
print("---")

Time elapsed CPU: 0.0011324463412165642
Calculated area: 5.999999999999997
Actual area: 6.0
Error one: 2.6645352591003757e-15
---


In [8]:
# Surface area of the sphere with cupy
# def identity_function_one(x):
#     return cp.ones(len(x))


# def identity_function_two(x):
#     return x[:, 0] ** 2 + x[:, 1] ** 2 + x[:, 2] ** 2


# integral_one = hull_surface_integral(identity_function_one, hull, use_gpu=True)
# integral_two = hull_surface_integral(identity_function_two, hull, use_gpu=True)

# print(f"Calculated area one: {integral_one}")
# print(f"Calculated area two: {integral_two}")
# print(f"Actual area: {hull.area}")
# print(f"Error one: {hull.area - integral_one}")
# print(f"Error two: {hull.area - integral_two}")

In [9]:
# Surface area of the sphere with cupy
# def function(x):
#     return x[:, 0] ** 2


# integral = hull_surface_integral(function, hull, use_gpu=True)

# print(f"Calculated: {integral}")
# print(f"Actual: {4*np.pi/3}")
# print(f"Error two: {4*np.pi/3 - integral}")